# Notebook 02: Feature Engineering

In this notebook, we stitch together the pipeline for processing our ML features:
1. **Load EEG** continuously -> Map to Epochs via events.tsv.
2. **Artifact Rejection** -> Drop noisy trials > ±100 µV.
3. **Extract EEG Features** -> Calculate Hjorth Parameters and Band Powers across frequency bands on clean epochs.
4. **Extract Audio Features** -> Get MFCC and Chroma coefficients from the `.mp3` files.
5. **Combine** -> Merge the EEG dataframe with the acoustic dataframe to form the final input `X`.

In [ ]:
import os
import sys
import pandas as pd

# Ensure our src modules can be imported
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

from eeg_utils import load_and_epoch_eeg, reject_artifacts
from features import extract_eeg_features_from_epochs
from audio_utils import extract_audio_features

base_path = os.path.abspath(os.path.join(os.getcwd(), '..'))

### 1. EEG Pipeline: Epoching & Artifact Rejection

In [ ]:
sub1_edf = os.path.join(base_path, 'data', 'sub-01', 'eeg', 'sub-01_task-run2_eeg.edf')
sub1_events = os.path.join(base_path, 'data', 'sub-01', 'eeg', 'sub-01_task-run2_events.tsv')

# 1. Load and epoch (0 to 15 seconds after stimulus onset)
epochs = load_and_epoch_eeg(sub1_edf, sub1_events, tmin=0.0, tmax=15.0)
print(f"Loaded {len(epochs)} epochs.")

# 2. Reject Artifacts (> ±100 µV)
clean_epochs = reject_artifacts(epochs, threshold_uv=100)
print(f"{len(clean_epochs)} epochs remaining after artifact rejection.")

### 2. Extract EEG Features

In [ ]:
# 3. Extract Hjorth and Band Powers for all remaining clean epochs
eeg_features_list = extract_eeg_features_from_epochs(clean_epochs)
eeg_df = pd.DataFrame(eeg_features_list)

display(eeg_df.head())

### 3. Audio Pipeline & Merging

In [ ]:
# 4. Extract Acoustic Features for an example MP3
example_audio_path = os.path.join(base_path, 'data', 'music', 'Set1', 'Set1', '005.mp3')

if os.path.exists(example_audio_path):
    audio_feat = extract_audio_features(example_audio_path)
    audio_feat['track_id'] = 5
    audio_df = pd.DataFrame([audio_feat])
    
    # 5. Merge Features based on Track ID
    combined_df = pd.merge(eeg_df, audio_df, on='track_id', how='inner')
    print(f"Combined Feature Matrix shape: {combined_df.shape}")
    display(combined_df.head())
else:
    print("Example audio path not found. Please ensure the MP3s are extracted.")